# 🌡️ Global Temperature EDA
> **Dataset:** Berkeley Earth Surface Temperature · 5 CSVs · 1750 – 2015  
> **Sections:** Setup → Data Loading → Preprocessing → Missing Values →  
> Global Trends → Seasonal Patterns → Country Analysis → City Analysis →  
> Distributions → Warming Analysis → Geo-Visualisation


## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
from src.data_loader import load_dataset
from src.preprocessing import preprocess

# Project modules
from src.data_loader   import load_dataset
from src.preprocessing import preprocess, missing_value_report
from src.analysis      import (
    summary_stats, correlation_matrix,
    global_yearly_trend, global_seasonal_trend,
    top_warming_countries, country_yearly_trend, country_monthly_profile,
    top_hottest_cities, top_coldest_cities, city_yearly_trend,
    temperature_distribution, monthly_boxplot_data,
)
from src.visualization import (
    plot_global_trend, plot_seasonal_trend,
    plot_top_warming_countries, plot_country_trend, plot_country_monthly_profile,
    plot_hottest_cities, plot_coldest_cities,
    plot_temperature_distribution, plot_monthly_boxplot,
    plot_correlation_heatmap, plot_temperature_map, plot_country_choropleth,
)

# Matplotlib dark style
plt.rcParams.update({
    'figure.facecolor': '#0E1117',
    'axes.facecolor':   '#1A1D2E',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#E0E0E0',
    'xtick.color':      '#A0A0B0',
    'ytick.color':      '#A0A0B0',
    'text.color':       '#E0E0E0',
    'grid.color':       '#2A2D3E',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'figure.dpi':       120,
    'font.family':      'sans-serif',
})
CYAN   = '#00D4FF'
RED    = '#FF6B6B'
AMBER  = '#FFD166'
GREEN  = '#55EFC4'
print('✅  All imports successful')


## 2. Data Loading

In [ ]:
# Load and preprocess all datasets
print('Loading global temperatures...')
g_df  = preprocess(load_dataset('global'))

print('Loading country temperatures...')
c_df  = preprocess(load_dataset('country'))

print('Loading major-city temperatures...')
mc_df = preprocess(load_dataset('major_city'))

print('Loading state temperatures...')
s_df  = preprocess(load_dataset('state'))

print('\n✅  All datasets loaded!')
print(f'  Global     : {g_df.shape}')
print(f'  Country    : {c_df.shape}')
print(f'  Major City : {mc_df.shape}')
print(f'  State      : {s_df.shape}')


In [ ]:
# Quick peek at each dataset
for name, df in [('Global', g_df), ('Country', c_df), ('Major City', mc_df)]:
    print(f'\n{'─'*60}')
    print(f'  {name}  —  columns: {list(df.columns)}')
    display(df.head(3))


## 3. Preprocessing & Missing Value Audit

In [ ]:
for name, df in [('Global', g_df), ('Country', c_df), ('Major City', mc_df), ('State', s_df)]:
    mv = missing_value_report(df)
    print(f'\n── {name} ─────────────────────────────')
    if mv.empty:
        print('   ✅ No missing values')
    else:
        display(mv)


In [ ]:
# Visualise missing-value % for major-city dataset
mv = missing_value_report(mc_df)
if not mv.empty:
    fig, ax = plt.subplots(figsize=(7, 3))
    mv['Missing %'].plot(kind='barh', ax=ax, color=RED)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values — Major City Dataset', color=CYAN)
    plt.tight_layout(); plt.show()


## 4. Descriptive Statistics

In [ ]:
print('=== Global Dataset ===')
display(summary_stats(g_df))


In [ ]:
print('=== Country Dataset ===')
display(summary_stats(c_df))


## 5. Global Land Temperature Trend

In [ ]:
yearly = global_yearly_trend(g_df)
rolling = yearly['MeanTemp'].rolling(10, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(yearly['year'], yearly['MeanTemp'], color=CYAN, lw=1.2, alpha=0.7, label='Annual Mean')
ax.plot(yearly['year'], rolling, color=AMBER, lw=2.5, linestyle='--', label='10-yr Rolling Avg')
ax.set_title('Global Land Average Temperature (1750–2015)', fontsize=14, color=CYAN)
ax.set_xlabel('Year'); ax.set_ylabel('°C')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Interactive Plotly version
plot_global_trend(yearly).show()


## 6. Seasonal Temperature Patterns

In [ ]:
seasonal = global_seasonal_trend(g_df)
print(seasonal.head())


In [ ]:
season_palette = {'Winter':'#74B9FF','Spring':'#55EFC4','Summer':'#FDCB6E','Autumn':'#E17055'}
fig, ax = plt.subplots(figsize=(13, 4))
for season, grp in seasonal.groupby('season'):
    ax.plot(grp['year'], grp['MeanTemp'], label=season,
            color=season_palette.get(season, CYAN), lw=1.5)
ax.set_title('Seasonal Temperature Trends', fontsize=14, color=CYAN)
ax.set_xlabel('Year'); ax.set_ylabel('°C')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Monthly climatology (all global records)
monthly_avg = g_df.groupby('month')['AverageTemperature'].mean()
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(MONTHS, monthly_avg.values,
              color=plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12)))
ax.set_title('Monthly Mean Temperature (Global)', fontsize=13, color=CYAN)
ax.set_ylabel('°C'); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()


## 7. Correlation Analysis

In [ ]:
corr = correlation_matrix(g_df)
print(corr)


In [ ]:
# Plotly heatmap
plot_correlation_heatmap(corr).show()


In [ ]:
# Seaborn version
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            linewidths=0.5, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Correlation Matrix — Global Dataset', color=CYAN)
plt.tight_layout(); plt.show()


## 8. Country-Level Analysis

In [ ]:
# Top warming countries
warming = top_warming_countries(c_df, n=15)
display(warming)


In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(warming)))
ax.barh(warming['Country'], warming['Warming'], color=colors[::-1])
ax.set_title('Countries with Greatest Temperature Rise', fontsize=13, color=CYAN)
ax.set_xlabel('Warming (°C)')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# Plotly interactive
plot_top_warming_countries(warming).show()


In [ ]:
# Single-country deep dive — India
country = 'India'
trend   = country_yearly_trend(c_df, country)
profile = country_monthly_profile(c_df, country)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(trend['year'], trend['MeanTemp'], color=CYAN, lw=1.8)
axes[0].set_title(f'{country} — Annual Trend', color=CYAN)
axes[0].set_xlabel('Year'); axes[0].set_ylabel('°C')
axes[0].grid(True)

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].bar([MONTHS[m-1] for m in profile['month']], profile['MeanTemp'],
            color=plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12)))
axes[1].set_title(f'{country} — Monthly Profile', color=CYAN)
axes[1].set_ylabel('°C'); axes[1].grid(axis='y', alpha=0.4)

plt.suptitle(f'{country} Temperature Analysis', color=AMBER, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Compare multiple countries — annual trend overlay
compare_countries = ['India', 'Russia', 'Brazil', 'United States', 'Australia']
COLORS = [CYAN, RED, AMBER, GREEN, '#B39DDB']

fig, ax = plt.subplots(figsize=(13, 5))
for country, color in zip(compare_countries, COLORS):
    tr = country_yearly_trend(c_df, country)
    if not tr.empty:
        ax.plot(tr['year'], tr['MeanTemp'].rolling(5).mean(),
                label=country, color=color, lw=2)
ax.set_title('Country Temperature Comparison (5-yr Smoothed)', fontsize=13, color=CYAN)
ax.set_xlabel('Year'); ax.set_ylabel('°C')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# World Choropleth
plot_country_choropleth(c_df).show()


## 9. City-Level Analysis

In [ ]:
hottest = top_hottest_cities(mc_df, n=15)
coldest = top_coldest_cities(mc_df, n=15)
print('Top 15 Hottest Cities:')
display(hottest)
print('\nTop 15 Coldest Cities:')
display(coldest)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(hottest['City'] + ', ' + hottest['Country'], hottest['MeanTemp'], color=RED, alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('☀️  Hottest Cities', color=CYAN); axes[0].set_xlabel('°C')
axes[0].grid(axis='x', alpha=0.4)

axes[1].barh(coldest['City'] + ', ' + coldest['Country'], coldest['MeanTemp'], color='#74B9FF', alpha=0.85)
axes[1].set_title('❄️  Coldest Cities', color=CYAN); axes[1].set_xlabel('°C')
axes[1].grid(axis='x', alpha=0.4)

plt.suptitle('City Temperature Rankings', color=AMBER, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# City trend — Bombay (Mumbai)
city   = 'Bombay'
c_trend = city_yearly_trend(mc_df, city)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(c_trend['year'], c_trend['MeanTemp'], color=CYAN, lw=1.8)
ax.plot(c_trend['year'], c_trend['MeanTemp'].rolling(10).mean(),
        color=AMBER, lw=2.5, linestyle='--', label='10-yr Rolling')
ax.set_title(f'{city} — Annual Temperature Trend', fontsize=13, color=CYAN)
ax.set_xlabel('Year'); ax.set_ylabel('°C')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# City scatter map
plot_temperature_map(mc_df).show()


## 10. Temperature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datasets_list = [('Global', g_df), ('Country', c_df), ('Major City', mc_df)]
for ax, (name, df) in zip(axes, datasets_list):
    series = temperature_distribution(df)
    ax.hist(series, bins=60, color=CYAN, alpha=0.75, edgecolor='#1A1D2E')
    ax.axvline(series.mean(), color=AMBER, linewidth=2, linestyle='--',
               label=f'Mean {series.mean():.1f}°C')
    ax.set_title(f'{name} Distribution', color=CYAN)
    ax.set_xlabel('°C'); ax.legend(); ax.grid(axis='y', alpha=0.4)

plt.suptitle('Temperature Distributions', color=AMBER, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Monthly box-plot
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
mb_data = monthly_boxplot_data(mc_df)

fig, ax = plt.subplots(figsize=(13, 5))
month_groups = [mb_data[mb_data['month'] == m]['AverageTemperature'].dropna() for m in range(1,13)]
bp = ax.boxplot(month_groups, patch_artist=True, labels=MONTHS,
                medianprops=dict(color=AMBER, linewidth=2))
for patch, color in zip(bp['boxes'],
                         plt.cm.RdYlBu_r(np.linspace(0.05, 0.95, 12))):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax.set_title('Monthly Temperature Distribution (Major Cities)', fontsize=13, color=CYAN)
ax.set_ylabel('°C'); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()


## 11. Warming Analysis — Pre vs Post 1950

In [ ]:
# Split era: pre vs post-1950
pre  = c_df[c_df['year'] < 1950]
post = c_df[c_df['year'] >= 1950]

pre_mean  = pre.groupby('Country')['AverageTemperature'].mean().rename('Pre_1950')
post_mean = post.groupby('Country')['AverageTemperature'].mean().rename('Post_1950')

era_df = pd.concat([pre_mean, post_mean], axis=1).dropna()
era_df['Warming'] = era_df['Post_1950'] - era_df['Pre_1950']
era_df = era_df.sort_values('Warming', ascending=False)

print('Countries that warmed the most (Pre vs Post 1950):')
display(era_df.head(15))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
top15 = era_df.head(15)
colors = plt.cm.Reds(np.linspace(0.4, 0.9, 15))
ax.barh(top15.index, top15['Warming'], color=colors[::-1])
ax.set_title('Temperature Increase: Post-1950 vs Pre-1950 (Top 15)', fontsize=13, color=CYAN)
ax.set_xlabel('Warming (°C)')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# Global annual mean — decade-level trend
decade_avg = g_df.groupby('decade')['AverageTemperature'].mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(decade_avg.index, decade_avg.values, width=8,
       color=plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(decade_avg))),
       edgecolor='#1A1D2E')
ax.set_title('Global Mean Temperature by Decade', fontsize=13, color=CYAN)
ax.set_xlabel('Decade'); ax.set_ylabel('°C')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()


## 12. US State-Level Patterns

In [ ]:
us_states = s_df[s_df['Country'] == 'United States']
if not us_states.empty:
    state_avg = us_states.groupby('State')['AverageTemperature'].mean().dropna().sort_values(ascending=False)
    print(f'{len(state_avg)} US states found')
    display(state_avg.head(10))


In [ ]:
if not us_states.empty:
    top_n  = state_avg.head(10)
    bot_n  = state_avg.tail(10)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].barh(top_n.index, top_n.values, color=RED, alpha=0.85)
    axes[0].invert_yaxis()
    axes[0].set_title('Hottest US States', color=CYAN); axes[0].set_xlabel('°C')
    axes[0].grid(axis='x', alpha=0.4)

    axes[1].barh(bot_n.index, bot_n.values, color='#74B9FF', alpha=0.85)
    axes[1].set_title('Coldest US States', color=CYAN); axes[1].set_xlabel('°C')
    axes[1].grid(axis='x', alpha=0.4)

    plt.suptitle('US State Temperature Rankings', color=AMBER, fontsize=14)
    plt.tight_layout(); plt.show()


## 13. 📝 Key Takeaways

| Finding | Detail |
|---------|--------|
| **Global warming trend** | Clear upward trend from ~1850, accelerating post-1950 |
| **Hottest decade** | 2000s consistently the warmest on record |
| **Seasonal spread** | Summer–Winter gap has narrowed slightly in many regions |
| **Most-warming countries** | Arctic/sub-arctic nations show greatest rise (Russia, Canada, etc.) |
| **Hottest cities** | Equatorial/desert cities (Djibouti, Niamey, etc.) |
| **Coldest cities** | High-latitude Siberian & Canadian cities |
| **Data coverage** | Pre-1850 records are sparse; uncertainty is higher |
| **Missing data** | Temperature uncertainty columns have the most gaps |
